# Tema 5 — Distribuição de Computadores por Aluno em Escolas Públicas

### Identificação do projeto
- **Disciplina:** Linguagens de Programação
- **Mês e ano:** Outubro de 2026
- **Professor(a):** Alexandre Louzada
- **Aluno(a):** Pedro Henriques Silva

> **Tema:** análise da distribuição de computadores por aluno e das desigualdades de infraestrutura tecnológica entre estados e entre escolas urbanas e rurais.

---

# 1. Título e Introdução

A infraestrutura tecnológica é um componente importante da capacidade das escolas públicas de oferecer atividades pedagógicas apoiadas por computadores. Entretanto, a disponibilidade de equipamentos pode variar entre estados e entre escolas localizadas em áreas urbanas e rurais.

Este projeto analisa dados de **2015 a 2024** para investigar como a quantidade de computadores se relaciona com o número de alunos e para identificar diferenças territoriais na disponibilidade relativa de equipamentos.

A análise é importante porque uma simples contagem de computadores não permite comparar adequadamente grupos de tamanhos diferentes. Por isso, será utilizada uma medida relativa — computadores por aluno — permitindo avaliar a disponibilidade de equipamentos em relação à demanda potencial.

# 2. Definição do Problema

## Pergunta principal

**Como evoluiu a disponibilidade de computadores por aluno nas escolas públicas entre 2015 e 2024, e quais desigualdades podem ser observadas entre estados e entre zonas urbana e rural?**

## Por que essa pergunta é relevante?

A quantidade absoluta de computadores pode ser enganosa: um estado pode possuir muitos equipamentos, mas também possuir uma quantidade muito maior de alunos. A relação entre computadores e alunos permite uma comparação mais justa da infraestrutura disponível.

Além disso, a comparação entre áreas urbanas e rurais permite verificar se a localização das escolas está associada a diferenças na disponibilidade relativa de equipamentos.

## Qual decisão poderia ser tomada com base nessa análise?

Os resultados podem apoiar decisões de **priorização de investimentos**, direcionando atenção para estados, anos ou tipos de escola que apresentem menor disponibilidade relativa de computadores ou maior desigualdade urbano-rural.

> **Importante:** os dados permitem identificar padrões de infraestrutura; eles não permitem, sozinhos, afirmar a qualidade pedagógica do uso dos computadores.

# 3. Definição do KPI

## KPI principal: Computadores por aluno

O indicador principal será:

**KPI = Número total de computadores ÷ Número total de alunos**

### Por que esse KPI foi escolhido?

Ele relaciona diretamente a oferta de equipamentos à quantidade de alunos. Assim, permite comparar estados, anos e zonas mesmo quando o tamanho das redes escolares é diferente.

### Como interpretar?

- **KPI maior:** maior disponibilidade relativa de computadores.
- **KPI menor:** menor disponibilidade relativa.
- Exemplo: KPI = 0,50 significa aproximadamente **1 computador para cada 2 alunos**, em termos agregados.

### Cuidados de interpretação

O KPI é uma medida de disponibilidade agregada. Ele não informa se os computadores estão funcionando, atualizados, conectados à internet ou distribuídos igualmente dentro das escolas.

# 4. Aquisição dos Dados

## Fonte e método de obtenção

**Fonte utilizada:** arquivo Excel fornecido para o projeto: `Simulacao_Computadores_por_Aluno_Escolas_Publicas_2015_2024 (1).xlsx`.

**Método de obtenção:** upload do arquivo para o ambiente do notebook e leitura com `pandas`.

A base contém registros por **estado, tipo de escola e ano**, abrangendo 2015–2024.

## Principais variáveis

| Variável | Descrição |
|---|---|
| Estado | Unidade federativa |
| Tipo de Escola | Urbana ou Rural |
| Ano | Ano da observação |
| Número de Escolas | Quantidade de escolas |
| Número de Alunos | Quantidade de alunos |
| Número de Computadores | Quantidade de computadores |
| Computadores por Aluno | Razão registrada na base |

A análise recalcula o KPI a partir dos totais de alunos e computadores para manter consistência nas agregações.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Arquivo de entrada
ARQUIVO = Path("Simulacao_Computadores_por_Aluno_Escolas_Publicas_2015_2024 (1).xlsx")

df = pd.read_excel(ARQUIVO)

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
display(df.head())

# 5. Limpeza e Preparação dos Dados

Nesta etapa serão verificados:
- valores nulos;
- tipos das variáveis;
- duplicidades;
- consistência dos valores numéricos;
- padronização dos nomes das colunas;
- criação do KPI utilizado na análise.

A limpeza é essencial porque erros de tipo ou valores ausentes podem alterar as estatísticas e os gráficos.

In [ ]:
# Cópia para preservar a base original
dados = df.copy()

# 1. Padronização dos nomes
dados.columns = [
    "estado",
    "tipo_escola",
    "ano",
    "numero_escolas",
    "numero_alunos",
    "numero_computadores",
    "computadores_por_aluno_original"
]

# 2. Conversão de tipos
dados["ano"] = pd.to_numeric(dados["ano"], errors="coerce").astype("Int64")

colunas_numericas = [
    "numero_escolas",
    "numero_alunos",
    "numero_computadores",
    "computadores_por_aluno_original"
]

for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

# 3. Diagnóstico de nulos
print("Valores nulos por coluna:")
display(dados.isna().sum().to_frame("quantidade_nulos"))

# 4. Diagnóstico de duplicidades
print("Linhas duplicadas:", dados.duplicated().sum())

# 5. Verificação de valores inválidos nos principais quantitativos
print("Alunos <= 0:", (dados["numero_alunos"] <= 0).sum())
print("Computadores < 0:", (dados["numero_computadores"] < 0).sum())

# 6. Como a base não apresenta nulos nas variáveis essenciais, não é necessário imputar valores.
# Linhas com informações essenciais ausentes seriam excluídas antes do cálculo do KPI.
dados = dados.dropna(subset=["estado", "tipo_escola", "ano", "numero_alunos", "numero_computadores"])

# 7. KPI recalculado
dados["kpi_computadores_aluno"] = (
    dados["numero_computadores"] / dados["numero_alunos"]
)

print("\nBase após preparação:", dados.shape)
display(dados.head())

## Verificação do KPI original versus KPI recalculado

A coluna original `Computadores por Aluno` é mantida para referência, mas o KPI utilizado no projeto é recalculado como computadores divididos por alunos. Isso permite agregá-lo corretamente por estado, ano e zona.

In [ ]:
comparacao = dados[[
    "computadores_por_aluno_original",
    "kpi_computadores_aluno"
]].describe()

display(comparacao.round(3))

# 6. Análise Exploratória

## 6.1 Estatísticas descritivas gerais

Primeiro observamos o tamanho da base e os principais valores de alunos, computadores e escolas.

In [ ]:
estatisticas = dados[[
    "numero_escolas",
    "numero_alunos",
    "numero_computadores",
    "kpi_computadores_aluno"
]].describe().T

display(estatisticas.round(3))

print("Estados:", dados["estado"].nunique())
print("Anos:", dados["ano"].min(), "a", dados["ano"].max())
print("Tipos de escola:", dados["tipo_escola"].unique())

## 6.2 Infraestrutura por estado

A comparação por estado utiliza os totais de alunos e computadores e depois calcula o KPI. Essa abordagem evita calcular uma simples média dos KPIs de registros individuais.

In [ ]:
por_estado = (
    dados.groupby("estado")[["numero_escolas", "numero_alunos", "numero_computadores"]]
    .sum()
)

por_estado["kpi_computadores_aluno"] = (
    por_estado["numero_computadores"] / por_estado["numero_alunos"]
)

por_estado = por_estado.sort_values("kpi_computadores_aluno", ascending=False)

display(por_estado.round(3))

## 6.3 Evolução temporal

A evolução anual mostra se a disponibilidade relativa de computadores aumentou ou diminuiu no período.

In [ ]:
por_ano = (
    dados.groupby("ano")[["numero_alunos", "numero_computadores"]]
    .sum()
)

por_ano["kpi_computadores_aluno"] = (
    por_ano["numero_computadores"] / por_ano["numero_alunos"]
)

display(por_ano.round(3))

ano_inicial = por_ano.index.min()
ano_final = por_ano.index.max()
kpi_inicial = por_ano.loc[ano_inicial, "kpi_computadores_aluno"]
kpi_final = por_ano.loc[ano_final, "kpi_computadores_aluno"]
variacao = (kpi_final / kpi_inicial - 1) * 100

print(f"KPI em {ano_inicial}: {kpi_inicial:.3f}")
print(f"KPI em {ano_final}: {kpi_final:.3f}")
print(f"Variação relativa entre o início e o fim: {variacao:.1f}%")

## 6.4 Comparação urbano × rural

A seguir, a razão é calculada separadamente para escolas urbanas e rurais em cada ano. O objetivo é verificar se a diferença entre as zonas é persistente ou varia ao longo do tempo.

In [ ]:
por_ano_zona = (
    dados.groupby(["ano", "tipo_escola"])[["numero_alunos", "numero_computadores"]]
    .sum()
)

por_ano_zona["kpi_computadores_aluno"] = (
    por_ano_zona["numero_computadores"] /
    por_ano_zona["numero_alunos"]
)

evolucao_zona = por_ano_zona["kpi_computadores_aluno"].unstack()

evolucao_zona["gap_urbano_rural"] = (
    evolucao_zona["Urbana"] - evolucao_zona["Rural"]
).abs()

display(evolucao_zona.round(3))

# 7. Visualizações

As visualizações foram selecionadas para responder diretamente à pergunta do projeto. Cada gráfico possui título, eixos nomeados e uma quantidade controlada de elementos visuais.

## 7.1 Evolução do KPI

Este gráfico responde: **como a disponibilidade relativa de computadores mudou ao longo do período?**

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    por_ano.index,
    por_ano["kpi_computadores_aluno"],
    marker="o",
    linewidth=2
)

plt.title("Evolução de computadores por aluno — 2015 a 2024")
plt.xlabel("Ano")
plt.ylabel("Computadores por aluno")
plt.xticks(por_ano.index)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7.2 Urbana × Rural ao longo do tempo

Este gráfico responde: **as escolas urbanas e rurais apresentam a mesma disponibilidade relativa de computadores?**

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    evolucao_zona.index,
    evolucao_zona["Urbana"],
    marker="o",
    linewidth=2,
    label="Urbana"
)

plt.plot(
    evolucao_zona.index,
    evolucao_zona["Rural"],
    marker="o",
    linewidth=2,
    label="Rural"
)

plt.title("Computadores por aluno: comparação urbana e rural")
plt.xlabel("Ano")
plt.ylabel("Computadores por aluno")
plt.xticks(evolucao_zona.index)
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 7.3 Diferença urbano-rural

Aqui não basta saber qual linha está acima. O foco é medir **o tamanho da desigualdade** entre as zonas em cada ano.

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    evolucao_zona.index.astype(str),
    evolucao_zona["gap_urbano_rural"]
)

plt.title("Diferença urbano-rural na razão de computadores por aluno")
plt.xlabel("Ano")
plt.ylabel("Diferença absoluta")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7.4 Comparação entre estados

Este gráfico responde: **quais estados apresentam maior e menor disponibilidade relativa de computadores?**

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    por_estado.index,
    por_estado["kpi_computadores_aluno"]
)

plt.title("Computadores por aluno por estado")
plt.xlabel("Estado")
plt.ylabel("Computadores por aluno")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7.5 Distribuição do KPI

O histograma ajuda a verificar a distribuição dos valores do indicador nos registros da base e se há concentração ou dispersão relevante.

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    dados["kpi_computadores_aluno"],
    bins=15,
    edgecolor="black"
)

plt.title("Distribuição da razão de computadores por aluno")
plt.xlabel("Computadores por aluno")
plt.ylabel("Frequência")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# 8. Insights e Interpretação

## 8.1 Estados

O ranking descritivo mostra que **CE apresenta a maior razão agregada de computadores por aluno, com aproximadamente 0,589**, enquanto **PR apresenta a menor, com aproximadamente 0,416**.

Isso representa uma diferença relevante na disponibilidade relativa de equipamentos. Entretanto, esse resultado não significa que CE possua necessariamente mais computadores em termos absolutos: o KPI considera também o número de alunos.

## 8.2 Evolução temporal

O KPI deve ser interpretado considerando simultaneamente a quantidade de computadores e a quantidade de alunos. Uma mudança no indicador pode ocorrer tanto por aumento/redução dos equipamentos quanto por alteração no número de alunos.

## 8.3 Desigualdade urbano-rural

Na base analisada, a maior diferença absoluta entre as razões urbana e rural ocorre em **2017, com aproximadamente 0,171**. Já a menor diferença ocorre em **2020, com aproximadamente 0,003**.

Isso indica que a desigualdade entre as zonas não permanece constante: há anos em que as razões se aproximam bastante e outros em que se afastam.

## 8.4 Evidências e limitações

As conclusões são sustentadas pelos agrupamentos por estado, ano e tipo de escola e pelo KPI calculado a partir dos totais.

### Limitações
- O KPI não informa se os computadores estão funcionando.
- Não há, neste indicador, informação sobre qualidade ou idade dos equipamentos.
- Não mede acesso à internet.
- A razão agregada pode esconder desigualdades entre escolas dentro do mesmo estado ou zona.
- A base permite analisar associação e distribuição, mas não estabelecer causalidade sobre os motivos das diferenças.

# 9. Conclusão

A análise mostra que a distribuição de computadores por aluno apresenta **variação entre estados e ao longo do tempo**, além de diferenças entre escolas urbanas e rurais.

A pergunta inicial pode ser respondida da seguinte forma:

> **A disponibilidade relativa de computadores não é uniforme. Existem diferenças entre estados e a distância entre zonas urbana e rural varia de ano para ano.**

Na base analisada:
- o **CE** possui a maior razão agregada de computadores por aluno (**0,589**);
- o **PR** possui a menor (**0,416**);
- a maior desigualdade urbano-rural ocorre em **2017 (0,171)**;
- a menor ocorre em **2020 (0,003)**.

## Possíveis decisões

Com base nesses resultados, gestores poderiam:
1. investigar as regiões com menor disponibilidade relativa;
2. avaliar a necessidade de priorização de investimentos;
3. acompanhar separadamente escolas urbanas e rurais;
4. repetir o indicador periodicamente para monitorar a evolução.

Essas decisões devem ser complementadas por informações sobre funcionamento, conectividade, idade dos equipamentos, quantidade de laboratórios e condições de uso.

# 10. Reflexão Final — Data Storytelling

## Se este projeto fosse apresentado a um gestor

### Principal mensagem

**A infraestrutura computacional apresenta desigualdades que não podem ser avaliadas apenas pelo número total de computadores. A relação entre equipamentos e alunos revela diferenças relevantes entre estados e entre zonas urbana e rural, e essas diferenças mudam ao longo do tempo.**

### Ação sugerida

A primeira ação seria utilizar o KPI como **indicador de monitoramento**, combinando-o com informações de qualidade dos equipamentos e conectividade. Estados ou zonas com menor disponibilidade relativa poderiam ser analisados com maior detalhe antes da definição de investimentos.

### Por que essa abordagem é importante?

Porque o objetivo não deve ser simplesmente aumentar o número de computadores, mas reduzir situações em que a quantidade de equipamentos é insuficiente em relação ao número de alunos e entender onde as desigualdades de infraestrutura são mais relevantes.

**Mensagem final:** os dados ajudam a transformar uma pergunta ampla sobre infraestrutura tecnológica em uma análise mensurável, comparável e útil para apoiar decisões.

# Anexo — Resumo automático dos resultados

A célula abaixo gera os principais indicadores do projeto automaticamente, evitando que números sejam digitados manualmente no relatório.

In [ ]:
estado_maior = por_estado["kpi_computadores_aluno"].idxmax()
estado_menor = por_estado["kpi_computadores_aluno"].idxmin()
ano_maior_gap = evolucao_zona["gap_urbano_rural"].idxmax()
ano_menor_gap = evolucao_zona["gap_urbano_rural"].idxmin()

resumo = pd.DataFrame({
    "Indicador": [
        "Período",
        "Estado com maior KPI",
        "Maior KPI",
        "Estado com menor KPI",
        "Menor KPI",
        "Ano de maior desigualdade urbano-rural",
        "Maior desigualdade",
        "Ano de menor desigualdade urbano-rural",
        "Menor desigualdade"
    ],
    "Resultado": [
        f"{por_ano.index.min()}–{por_ano.index.max()}",
        estado_maior,
        round(por_estado.loc[estado_maior, "kpi_computadores_aluno"], 3),
        estado_menor,
        round(por_estado.loc[estado_menor, "kpi_computadores_aluno"], 3),
        int(ano_maior_gap),
        round(evolucao_zona.loc[ano_maior_gap, "gap_urbano_rural"], 3),
        int(ano_menor_gap),
        round(evolucao_zona.loc[ano_menor_gap, "gap_urbano_rural"], 3)
    ]
})

display(resumo)